In [1]:
%cd ~/galaxy_evolution_project/full_desi_data

/home/qarni/galaxy_evolution_project/full_desi_data


/home/qarni/miniforge3/envs/galaxy_env/lib/python3.10/site-packages/IPython/core/magics/osm.py:393: UserWarning: This is now an optional IPython functionality, using bookmarks requires you to install the `pickleshare` library.
  bkms = self.shell.db.get('bookmarks', {})
/home/qarni/miniforge3/envs/galaxy_env/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
# =====================================================
#  STAGE 1 — DESI DR1 (ELG + LRG): Clean & Convert
# =====================================================
import pandas as pd, numpy as np
from astropy.io import fits
from astropy.cosmology import Planck18 as cosmo
import astropy.units as u

# -----------------------------------------------------
# Helper: read DESI .fits (for galaxies or randoms)
# -----------------------------------------------------
def read_basic(fits_path, nz_path=None, nmax=None):
    with fits.open(fits_path) as hdul:
        data = hdul[1].data
        cols = data.columns.names
        ra = data["RA"]
        dec = data["DEC"]

        # --- Detect redshift source ---
        if "Z_not4clus" in cols:
            z = data["Z_not4clus"]
            print(f"✅ Using Z_not4clus from {fits_path}")
        elif "Z" in cols:
            z = data["Z"]
            print(f"✅ Using Z from {fits_path}")
        elif nz_path is not None:
            # --- Assign random z based on n(z) distribution file ---
            print(f"⚙️  No Z column, sampling from n(z) in {nz_path}")
            nz = np.loadtxt(nz_path)
            z_bins, nz_counts = nz[:,0], nz[:,1]
            z_probs = nz_counts / nz_counts.sum()
            z = np.random.choice(z_bins, size=len(ra), p=z_probs)
        else:
            raise KeyError(f"No redshift info found for {fits_path}")

        df = pd.DataFrame({"RA": ra, "DEC": dec, "Z": z})
        if nmax:
            df = df.sample(nmax, random_state=42)

    print(f"✅ Loaded {len(df):,} rows from {fits_path.split('/')[-1]}")
    return df


# -----------------------------------------------------
# Load ELG + LRG galaxy and random catalogs
# -----------------------------------------------------
elg_gal = read_basic("ELG_LOPnotqso_full_HPmapcut.dat.fits")
elg_ran = read_basic("ELG_LOPnotqso_0_full_HPmapcut.ran.fits",
                     nz_path="ELG_LOPnotqso_full_HPmapcut_nz.txt")
lrg_gal = read_basic("LRG_full_HPmapcut.dat.fits")
lrg_ran = read_basic("LRG_0_full_HPmapcut.ran.fits",
                     nz_path="LRG_full_HPmapcut_nz.txt")

# -----------------------------------------------------
# Merge both samples and compute α normalization
# -----------------------------------------------------
df_gal = pd.concat([elg_gal.assign(SAMPLE="ELG"),
                    lrg_gal.assign(SAMPLE="LRG")], ignore_index=True)
df_ran = pd.concat([elg_ran.assign(SAMPLE="ELG"),
                    lrg_ran.assign(SAMPLE="LRG")], ignore_index=True)

alpha = len(df_gal) / len(df_ran)
print(f"\nNormalization factor α = {alpha:.3e}")
print(f"{len(df_gal):,} galaxies  |  {len(df_ran):,} randoms")

# -----------------------------------------------------
# Convert RA, DEC, z → Cartesian coordinates
# -----------------------------------------------------
def add_cartesian(df):
    chi = cosmo.comoving_distance(df["Z"]).to(u.Mpc).value
    ra, dec = np.deg2rad(df["RA"]), np.deg2rad(df["DEC"])
    df["x"] = chi * np.cos(dec) * np.cos(ra)
    df["y"] = chi * np.cos(dec) * np.sin(ra)
    df["z"] = chi * np.sin(dec)
    return df

df_gal = add_cartesian(df_gal)
df_ran = add_cartesian(df_ran)

# -----------------------------------------------------
# Save cleaned catalogs
# -----------------------------------------------------
df_gal.to_csv("desi_galaxies_xyz.csv", index=False)
df_ran.to_csv("desi_randoms_xyz.csv", index=False)
print("\n✅ Saved → desi_galaxies_xyz.csv & desi_randoms_xyz.csv")
print(df_gal.head())

✅ Using Z_not4clus from ELG_LOPnotqso_full_HPmapcut.dat.fits
✅ Loaded 11,019,548 rows from ELG_LOPnotqso_full_HPmapcut.dat.fits
⚙️  No Z column, sampling from n(z) in ELG_LOPnotqso_full_HPmapcut_nz.txt
✅ Loaded 14,811,559 rows from ELG_LOPnotqso_0_full_HPmapcut.ran.fits
✅ Using Z_not4clus from LRG_full_HPmapcut.dat.fits
✅ Loaded 3,461,048 rows from LRG_full_HPmapcut.dat.fits
⚙️  No Z column, sampling from n(z) in LRG_full_HPmapcut_nz.txt
✅ Loaded 14,350,911 rows from LRG_0_full_HPmapcut.ran.fits

Normalization factor α = 4.965e-01
14,480,596 galaxies  |  29,162,470 randoms


/home/qarni/miniforge3/envs/galaxy_env/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:2605: RuntimeWarning: invalid value encountered in _integral_comoving_distance_z1z2_scalar (vectorized)
  outputs = ufunc(*inputs)



✅ Saved → desi_galaxies_xyz.csv & desi_randoms_xyz.csv
          RA        DEC         Z SAMPLE            x            y           z
0  21.701752 -19.385530       NaN    ELG     0.000000     0.000000    -0.00000
1  21.713325 -19.406004       NaN    ELG     0.000000     0.000000    -0.00000
2  21.723061 -19.415860       NaN    ELG     0.000000     0.000000    -0.00000
3  21.726791 -19.412397       NaN    ELG     0.000000     0.000000    -0.00000
4  21.735261 -19.411350  1.086749    ELG  3159.094326  1259.408932 -1198.39639


In [1]:
import numpy as np, pandas as pd, h5py, os
from scipy.ndimage import gaussian_filter

gal = pd.read_csv("desi_galaxies_xyz.csv")
ran = pd.read_csv("desi_randoms_xyz.csv")
alpha = len(gal) / len(ran)
print(f"Loaded {len(gal):,} galaxies, {len(ran):,} randoms (α = {alpha:.3e})")

box_size = 640.0
grid_N   = 256

def clip_box(df, box_size):
    for c in ['x','y','z']:
        df[c] = np.clip(df[c], -box_size/2, box_size/2)
    return df

gal, ran = clip_box(gal, box_size), clip_box(ran, box_size)
for c in ['x','y','z']:
    gal[c] += box_size/2
    ran[c] += box_size/2

edges = np.linspace(0, box_size, grid_N+1)
n_gal, _ = np.histogramdd(gal[['x','y','z']].values, bins=(edges,)*3)
n_ran, _ = np.histogramdd(ran[['x','y','z']].values, bins=(edges,)*3)
delta = (n_gal - alpha*n_ran)/(alpha*n_ran + 1e-10)
delta_smooth = gaussian_filter(delta, sigma=2.0)

os.makedirs("output", exist_ok=True)
with h5py.File("output/density_field.h5", "w") as f:
    f.create_dataset("delta", data=delta_smooth)
    f.attrs["box_size"] = box_size
    f.attrs["grid_N"]   = grid_N
    f.attrs["R_smooth"] = 2.0

print("✅ Re-saved density_field.h5 with correct attributes.")

FileNotFoundError: [Errno 2] No such file or directory: 'desi_galaxies_xyz.csv'

In [2]:
ls


 01_DESI_DR1_DataPreparation.ipynb  'desi data.ipynb'  'full data.ipynb'
